# Data Preprocessing

Leakage-safe preprocessing after null imputation. Empty/duplicate/constant column rules and IQR bounds are learned from `train_split_imputed` only, then applied to validation and test. Raw source files are not modified.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

## Load Imputed Split Data

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"

train = pd.read_csv("data/train_split_imputed.csv")
val = pd.read_csv("data/val_split_imputed.csv")
test = pd.read_csv("data/test_imputed.csv")

print("train split imputed shape:", train.shape)
print("validation split imputed shape:", val.shape)
print("test imputed shape:", test.shape)

train split imputed shape: (552070, 15)
validation split imputed shape: (138018, 15)
test imputed shape: (295753, 14)


In [3]:
feature_cols = [col for col in train.columns if col not in [ID_COL, TARGET_COL]]
print("feature columns:", feature_cols)

feature columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## Empty Columns And Rows

A row is treated as empty if every non-ID, non-target value is missing. Rules are inspected across datasets, but removals are based on train only.

In [4]:
def non_id_target_cols(df):
    return [col for col in df.columns if col not in [ID_COL, TARGET_COL]]

train_empty_cols = train.columns[train.isna().all()].tolist()
val_empty_cols = val.columns[val.isna().all()].tolist()
test_empty_cols = test.columns[test.isna().all()].tolist()

train_empty_rows_mask = train[non_id_target_cols(train)].isna().all(axis=1)
val_empty_rows_mask = val[non_id_target_cols(val)].isna().all(axis=1)
test_empty_rows_mask = test[non_id_target_cols(test)].isna().all(axis=1)

empty_report = pd.DataFrame([
    {"dataset": "train_split", "empty_columns": train_empty_cols, "empty_row_count": int(train_empty_rows_mask.sum())},
    {"dataset": "val_split", "empty_columns": val_empty_cols, "empty_row_count": int(val_empty_rows_mask.sum())},
    {"dataset": "test", "empty_columns": test_empty_cols, "empty_row_count": int(test_empty_rows_mask.sum())},
])

empty_report

,dataset,empty_columns,empty_row_count
0,train_split,[],0
1,val_split,[],0
2,test,[],0


## Duplicate Rows

In [5]:
train_exact_duplicate_rows = int(train.duplicated().sum())
val_exact_duplicate_rows = int(val.duplicated().sum())
test_exact_duplicate_rows = int(test.duplicated().sum())

train_duplicate_data_rows_mask = train.duplicated(subset=[col for col in train.columns if col != ID_COL], keep="first")
val_duplicate_data_rows_mask = val.duplicated(subset=[col for col in val.columns if col != ID_COL], keep="first")
test_duplicate_data_rows_mask = test.duplicated(subset=[col for col in test.columns if col != ID_COL], keep="first")

duplicate_row_report = pd.DataFrame([
    {"dataset": "train_split", "exact_duplicate_rows": train_exact_duplicate_rows, "duplicate_rows_ignoring_id": int(train_duplicate_data_rows_mask.sum())},
    {"dataset": "val_split", "exact_duplicate_rows": val_exact_duplicate_rows, "duplicate_rows_ignoring_id": int(val_duplicate_data_rows_mask.sum())},
    {"dataset": "test", "exact_duplicate_rows": test_exact_duplicate_rows, "duplicate_rows_ignoring_id": int(test_duplicate_data_rows_mask.sum())},
])

duplicate_row_report

,dataset,exact_duplicate_rows,duplicate_rows_ignoring_id
0,train_split,0,0
1,val_split,0,0
2,test,0,0


## Duplicate Columns

In [6]:
def find_duplicate_columns(df):
    duplicate_groups = []
    used = set()

    for i, col in enumerate(df.columns):
        if col in used:
            continue
        group = [col]
        for other_col in df.columns[i + 1:]:
            if other_col in used:
                continue
            if df[col].equals(df[other_col]):
                group.append(other_col)
        if len(group) > 1:
            duplicate_groups.append(group)
            used.update(group)

    return duplicate_groups


def columns_to_drop_from_duplicate_groups(duplicate_groups, protected_cols):
    drops = []
    for group in duplicate_groups:
        keep = next((col for col in group if col in protected_cols), group[0])
        drops.extend([col for col in group if col != keep])
    return drops

train_duplicate_column_groups = find_duplicate_columns(train)
val_duplicate_column_groups = find_duplicate_columns(val)
test_duplicate_column_groups = find_duplicate_columns(test)

duplicate_column_report = pd.DataFrame([
    {"dataset": "train_split", "duplicate_column_groups": train_duplicate_column_groups},
    {"dataset": "val_split", "duplicate_column_groups": val_duplicate_column_groups},
    {"dataset": "test", "duplicate_column_groups": test_duplicate_column_groups},
])

duplicate_column_report

,dataset,duplicate_column_groups
0,train_split,[]
1,val_split,[]
2,test,[]


## Constant Columns

In [7]:
train_constant_cols = [
    col for col in train.columns
    if col not in [ID_COL, TARGET_COL] and train[col].nunique(dropna=True) <= 1
]
val_constant_cols = [
    col for col in val.columns
    if col not in [ID_COL, TARGET_COL] and val[col].nunique(dropna=True) <= 1
]
test_constant_cols = [
    col for col in test.columns
    if col != ID_COL and test[col].nunique(dropna=True) <= 1
]

constant_column_report = pd.DataFrame([
    {"dataset": "train_split", "constant_columns": train_constant_cols},
    {"dataset": "val_split", "constant_columns": val_constant_cols},
    {"dataset": "test", "constant_columns": test_constant_cols},
])

constant_column_report

,dataset,constant_columns
0,train_split,[]
1,val_split,[]
2,test,[]


## Apply Basic Cleaning

Train-derived column removals are applied to train/validation/test. Empty and duplicate row removals are applied only to train split; validation and test rows are preserved.

In [8]:
protected_cols = {ID_COL, TARGET_COL}
train_duplicate_cols_to_drop = columns_to_drop_from_duplicate_groups(
    train_duplicate_column_groups,
    protected_cols=protected_cols,
)

columns_to_drop_from_train_rules = sorted(set(
    col for col in train_empty_cols + train_duplicate_cols_to_drop + train_constant_cols
    if col not in protected_cols
))

train_preprocessed = train.copy()
val_preprocessed = val.copy()
test_preprocessed = test.copy()

for frame_name, frame in [
    ("train", train_preprocessed),
    ("val", val_preprocessed),
    ("test", test_preprocessed),
]:
    drop_cols = [col for col in columns_to_drop_from_train_rules if col in frame.columns]
    if drop_cols:
        frame.drop(columns=drop_cols, inplace=True)

train_preprocessed = train_preprocessed.loc[~train_empty_rows_mask].copy()
train_preprocessed = train_preprocessed.loc[~train_duplicate_data_rows_mask].copy()

basic_cleaning_report = pd.DataFrame([
    {"step": "drop train-derived empty/duplicate/constant columns", "removed": columns_to_drop_from_train_rules},
    {"step": "drop empty train rows", "removed": int(train_empty_rows_mask.sum())},
    {"step": "drop duplicate train rows ignoring id", "removed": int(train_duplicate_data_rows_mask.sum())},
    {"step": "keep validation rows", "removed": 0},
    {"step": "keep test rows", "removed": 0},
])

print("train shape before basic cleaning:", train.shape)
print("train shape after basic cleaning:", train_preprocessed.shape)
print("val shape before basic cleaning:", val.shape)
print("val shape after basic cleaning:", val_preprocessed.shape)
print("test shape before basic cleaning:", test.shape)
print("test shape after basic cleaning:", test_preprocessed.shape)

basic_cleaning_report

train shape before basic cleaning: (552070, 15)
train shape after basic cleaning: (552070, 15)
val shape before basic cleaning: (138018, 15)
val shape after basic cleaning: (138018, 15)
test shape before basic cleaning: (295753, 14)
test shape after basic cleaning: (295753, 14)


,step,removed
0,drop train-derived empty/duplicate/constant columns,[]
1,drop empty train rows,0
2,drop duplicate train rows ignoring id,0
3,keep validation rows,0
4,keep test rows,0


## IQR Outlier Flags

IQR bounds are learned from the cleaned train split only. Rows are not removed; outlier flags are added to train/validation/test using the same train-derived bounds.

In [9]:
numeric_feature_cols = [
    col for col in train_preprocessed.columns
    if col not in [ID_COL, TARGET_COL] and pd.api.types.is_numeric_dtype(train_preprocessed[col])
]

IQR_MULTIPLIER = 1.5

iqr_bounds_rows = []
for col in numeric_feature_cols:
    q1 = train_preprocessed[col].quantile(0.25)
    q3 = train_preprocessed[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr
    iqr_bounds_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
    })

iqr_bounds = pd.DataFrame(iqr_bounds_rows)
iqr_bounds

,column,q1,q3,iqr,lower_bound,upper_bound
0,sleep_duration,6.30,7.68,1.38,4.230,9.750
1,heart_rate,69.50,80.60,11.10,52.850,97.250
2,bmi,21.38,24.63,3.25,16.505,29.505
3,calorie_expenditure,2068.00,2438.00,370.00,1513.000,2993.000
4,step_count,5392.00,12089.00,6697.00,-4653.500,22134.500
5,exercise_duration,29.10,49.40,20.30,-1.350,79.850
6,water_intake,1.87,2.47,0.60,0.970,3.370


In [10]:
def add_iqr_flags(frame, bounds):
    frame = frame.copy()
    any_outlier = pd.Series(False, index=frame.index)
    per_column_counts = {}

    for row in bounds.itertuples(index=False):
        flag_col = f"{row.column}_iqr_outlier"
        col_outliers = (
            frame[row.column].notna()
            & ((frame[row.column] < row.lower_bound) | (frame[row.column] > row.upper_bound))
        )
        frame[flag_col] = col_outliers.astype(int)
        any_outlier |= col_outliers
        per_column_counts[row.column] = int(col_outliers.sum())

    frame["any_iqr_outlier"] = any_outlier.astype(int)
    return frame, any_outlier, per_column_counts


train_preprocessed, train_outlier_mask, train_outlier_counts = add_iqr_flags(train_preprocessed, iqr_bounds)
val_preprocessed, val_outlier_mask, val_outlier_counts = add_iqr_flags(val_preprocessed, iqr_bounds)
test_preprocessed, test_outlier_mask, test_outlier_counts = add_iqr_flags(test_preprocessed, iqr_bounds)

outlier_report = pd.DataFrame([
    {
        "column": row.column,
        "flag_column": f"{row.column}_iqr_outlier",
        "train_outlier_count": train_outlier_counts[row.column],
        "train_outlier_pct": train_outlier_counts[row.column] / len(train_preprocessed) * 100 if len(train_preprocessed) else 0,
        "val_outlier_count_using_train_bounds": val_outlier_counts[row.column],
        "val_outlier_pct_using_train_bounds": val_outlier_counts[row.column] / len(val_preprocessed) * 100 if len(val_preprocessed) else 0,
        "test_outlier_count_using_train_bounds": test_outlier_counts[row.column],
        "test_outlier_pct_using_train_bounds": test_outlier_counts[row.column] / len(test_preprocessed) * 100 if len(test_preprocessed) else 0,
    }
    for row in iqr_bounds.itertuples(index=False)
]).sort_values("train_outlier_count", ascending=False)

print("train rows flagged by IQR, not removed:", int(train_outlier_mask.sum()))
print("val rows flagged by train IQR bounds, not removed:", int(val_outlier_mask.sum()))
print("test rows flagged by train IQR bounds, not removed:", int(test_outlier_mask.sum()))
print("final train_preprocessed shape:", train_preprocessed.shape)
print("final val_preprocessed shape:", val_preprocessed.shape)
print("final test_preprocessed shape:", test_preprocessed.shape)

outlier_report.style.format({
    "train_outlier_pct": "{:.2f}",
    "val_outlier_pct_using_train_bounds": "{:.2f}",
    "test_outlier_pct_using_train_bounds": "{:.2f}",
})

train rows flagged by IQR, not removed: 51503
val rows flagged by train IQR bounds, not removed: 12875
test rows flagged by train IQR bounds, not removed: 27517
final train_preprocessed shape: (552070, 23)
final val_preprocessed shape: (138018, 23)
final test_preprocessed shape: (295753, 22)


,column,flag_column,train_outlier_count,train_outlier_pct,val_outlier_count_using_train_bounds,val_outlier_pct_using_train_bounds,test_outlier_count_using_train_bounds,test_outlier_pct_using_train_bounds
3,calorie_expenditure,calorie_expenditure_iqr_outlier,16480,2.99,4094,2.97,8841,2.99
6,water_intake,water_intake_iqr_outlier,15998,2.90,4028,2.92,8696,2.94
0,sleep_duration,sleep_duration_iqr_outlier,13145,2.38,3271,2.37,7104,2.40
2,bmi,bmi_iqr_outlier,5007,0.91,1284,0.93,2623,0.89
1,heart_rate,heart_rate_iqr_outlier,2766,0.50,675,0.49,1183,0.40
5,exercise_duration,exercise_duration_iqr_outlier,81,0.01,22,0.02,31,0.01
4,step_count,step_count_iqr_outlier,0,0.00,0,0.00,0,0.00


## Final Checks

In [11]:
def final_row(dataset_name, frame):
    subset_cols = [col for col in frame.columns if col != ID_COL]
    return {
        "dataset": dataset_name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "empty_rows_non_id": int(frame[subset_cols].isna().all(axis=1).sum()),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "duplicate_rows_ignoring_id": int(frame.duplicated(subset=[col for col in frame.columns if col != ID_COL], keep="first").sum()),
    }

final_check = pd.DataFrame([
    final_row("train_split_preprocessed", train_preprocessed),
    final_row("val_split_preprocessed", val_preprocessed),
    final_row("test_preprocessed", test_preprocessed),
])

final_check

,dataset,rows,columns,missing_values,empty_rows_non_id,exact_duplicate_rows,duplicate_rows_ignoring_id
0,train_split_preprocessed,552070,23,0,0,0,0
1,val_split_preprocessed,138018,23,0,0,0,0
2,test_preprocessed,295753,22,0,0,0,0


## Save Preprocessed Splits

In [12]:
train_preprocessed_path = "data/train_split_preprocessed.csv"
val_preprocessed_path = "data/val_split_preprocessed.csv"
test_preprocessed_path = "data/test_preprocessed.csv"

train_preprocessed.to_csv(train_preprocessed_path, index=False)
val_preprocessed.to_csv(val_preprocessed_path, index=False)
test_preprocessed.to_csv(test_preprocessed_path, index=False)

print("saved:", train_preprocessed_path, train_preprocessed.shape)
print("saved:", val_preprocessed_path, val_preprocessed.shape)
print("saved:", test_preprocessed_path, test_preprocessed.shape)

saved: data/train_split_preprocessed.csv (552070, 23)
saved: data/val_split_preprocessed.csv (138018, 23)
saved: data/test_preprocessed.csv (295753, 22)


In [13]:
train_preprocessed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier
0,313415,at-risk,7.17,91.3,26.86,2635.0,1398.0,49.1,2.03,non-veg,medium,average,sedentary,yes,male,0,0,0,0,0,0,0,0
1,3515,at-risk,6.99,75.1,24.23,2382.0,13466.0,50.8,1.83,balanced,low,average,active,occasional,male,0,0,0,0,0,0,0,0
2,501194,at-risk,8.66,82.4,21.41,2314.0,9473.0,23.3,3.09,balanced,low,poor,sedentary,yes,female,0,0,0,0,0,0,0,0
3,303602,at-risk,6.99,74.5,22.59,2165.0,7052.0,21.6,1.92,veg,medium,average,sedentary,occasional,male,0,0,0,0,0,0,0,0
4,117943,at-risk,8.83,68.2,22.01,2108.0,13521.0,52.9,2.35,non-veg,medium,average,active,yes,male,0,0,0,0,0,0,0,0


In [14]:
val_preprocessed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier
0,304516,at-risk,6.82,67.9,27.17,2464.0,13456.0,39.9,2.05,veg,medium,good,moderate,no,other,0,0,0,0,0,0,0,0
1,165358,at-risk,7.97,92.9,16.86,2240.0,7456.0,26.9,2.14,balanced,high,good,moderate,no,female,0,0,0,0,0,0,0,0
2,671841,at-risk,6.99,75.1,22.18,2352.0,4140.0,19.7,2.33,non-veg,low,poor,sedentary,occasional,male,0,0,0,0,0,0,0,0
3,403460,at-risk,6.99,83.1,21.93,2620.0,11656.0,41.1,1.22,non-veg,medium,good,moderate,occasional,other,0,0,0,0,0,0,0,0
4,351133,at-risk,8.14,80.0,23.41,2442.0,4484.0,38.9,2.32,veg,medium,average,sedentary,no,other,0,0,0,0,0,0,0,0


In [15]:
test_preprocessed.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male,0,0,0,0,0,0,0,0
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other,0,0,0,0,0,0,0,0
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,male,0,0,0,1,0,0,0,1
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other,0,0,0,0,0,0,0,0
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other,0,0,0,0,0,0,0,0
